In [ ]:
[markdown]
# # Pre-Entrega 1 — Extracción, Validación y Análisis Preliminar del Dataset
# **Analítica Descriptiva | ITBA 2026 — Real Estate Analytics**
#
# Este notebook documenta:
# 1. La carga y validación del dataset extraído de Argenprop
# 2. La auditoría de calidad de datos (nulos, outliers, inconsistencias)
# 3. El cálculo preliminar de KPIs
# 4. La visualización exploratoria inicial
# 5. Las mejoras implementadas sobre el scraper base


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

# Configuración visual
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'
sns.set_style("whitegrid")
PALETTE = ['#1B4F72', '#2471A3', '#2ECC71', '#F39C12', '#E74C3C', '#8E44AD', '#1ABC9C']


In [ ]:
[markdown]
# ---
# ## 1. Carga del Dataset


In [ ]:
df = pd.read_csv("../data/raw/dataset_argenprop_completo.csv")
print(f"✓ Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")


In [ ]:
df.head(3)


In [ ]:
[markdown]
# ---
# ## 2. Parsing y Normalización de Variables Crudas
#
# Las columnas `Precio` y `Expensas` vienen como texto. Las parseamos a numéricas.


In [ ]:
def parse_precio_usd(precio):
    """Extrae valor numérico de precios en USD."""
    if pd.isna(precio):
        return np.nan
    m = re.search(r'USD\s?([\d.]+)', str(precio))
    if m:
        return float(m.group(1).replace('.', ''))
    return np.nan

def parse_expensas_ars(expensas):
    """Extrae valor numérico de expensas en ARS."""
    if pd.isna(expensas):
        return np.nan
    texto = str(expensas).replace('+', '').strip()
    m = re.search(r'\$?\s?([\d.]+)', texto)
    if m:
        return float(m.group(1).replace('.', ''))
    return np.nan

df['precio_usd'] = df['Precio'].apply(parse_precio_usd)
df['expensas_ars'] = df['Expensas'].apply(parse_expensas_ars)

# Precio por m²
df['precio_m2'] = df['precio_usd'] / df['Sup_Total_m2']
df['precio_m2'] = df['precio_m2'].replace([np.inf, -np.inf], np.nan)

print(f"Precios parseados a USD: {df['precio_usd'].notna().sum():,} ({df['precio_usd'].notna().mean()*100:.1f}%)")
print(f"Expensas parseadas a ARS: {df['expensas_ars'].notna().sum():,} ({df['expensas_ars'].notna().mean()*100:.1f}%)")
print(f"Precio/m² calculable: {df['precio_m2'].notna().sum():,} ({df['precio_m2'].notna().mean()*100:.1f}%)")


In [ ]:
[markdown]
# ---
# ## 3. Auditoría de Calidad de Datos


In [ ]:
[markdown]
# ### 3.1 Análisis de Valores Nulos


In [ ]:
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(1)
nulos_df = pd.DataFrame({'Nulos': nulos, '% del total': nulos_pct})
nulos_df = nulos_df[nulos_df['Nulos'] > 0].sort_values('Nulos', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#E74C3C' if p > 30 else '#F39C12' if p > 10 else '#2ECC71' for p in nulos_df['% del total']]
bars = ax.barh(range(len(nulos_df)), nulos_df['% del total'], color=colors, edgecolor='white')
ax.set_yticks(range(len(nulos_df)))
ax.set_yticklabels(nulos_df.index, fontsize=10)
ax.set_xlabel("% de valores nulos")
ax.set_title("Proporción de valores nulos por variable")
ax.invert_yaxis()
ax.axvline(x=30, color='red', linestyle='--', alpha=0.4, label='Umbral crítico (30%)')
ax.legend(fontsize=9)
for i, (val, count) in enumerate(zip(nulos_df['% del total'], nulos_df['Nulos'])):
    ax.text(val + 0.5, i, f'{val}% ({count:,})', va='center', fontsize=8)
plt.tight_layout()
plt.show()

print("\nResumen: las variables más comprometidas son Antigüedad (38.3%) y Expensas (38.5%).")
print("Estas requerirán estrategias de imputación en la Pre-Entrega 2.")


In [ ]:
[markdown]
# ### 3.2 Detección de Outliers en Coordenadas Geográficas
#
# CABA se ubica aprox. en latitud [-34.70, -34.53] y longitud [-58.53, -58.33].
# Verificamos si hay coordenadas fuera de rango.


In [ ]:
# Límites geográficos de CABA (con margen)
LAT_MIN, LAT_MAX = -34.72, -34.52
LON_MIN, LON_MAX = -58.54, -58.33

coords_ok = df['Latitud'].between(LAT_MIN, LAT_MAX) & df['Longitud'].between(LON_MIN, LON_MAX)
coords_na = df['Latitud'].isna() | df['Longitud'].isna()
coords_bad = ~coords_ok & ~coords_na

print(f"Coordenadas dentro de CABA:  {coords_ok.sum():,} ({coords_ok.mean()*100:.1f}%)")
print(f"Coordenadas fuera de CABA:   {coords_bad.sum():,} ({coords_bad.mean()*100:.1f}%)")
print(f"Sin coordenadas:             {coords_na.sum():,} ({coords_na.mean()*100:.1f}%)")

if coords_bad.sum() > 0:
    print(f"\n⚠ Se detectaron {coords_bad.sum()} registros con coordenadas fuera de CABA.")
    print("Ejemplos de coordenadas anómalas:")
    print(df.loc[coords_bad, ['Barrio', 'Calle', 'Latitud', 'Longitud']].head(5).to_string())
    print("\nEstas se tratarán como errores de geocodificación en la fase de limpieza.")


In [ ]:
[markdown]
# ### 3.3 Detección de Outliers en Antigüedad


In [ ]:
ant = df['Antiguedad'].dropna()
print(f"Antigüedad - Media: {ant.mean():.1f} | Mediana: {ant.median():.1f} | Max: {ant.max():.0f}")

outliers_ant = ant[ant > 150]
print(f"\nRegistros con antigüedad > 150 años: {len(outliers_ant)}")
if len(outliers_ant) > 0:
    print("Valores detectados:", sorted(outliers_ant.unique())[:10])
    print("⚠ El valor 2026 indica que se ingresó el AÑO de construcción en vez de los años de antigüedad.")


In [ ]:
[markdown]
# ### 3.4 Detección de Outliers en Precios


In [ ]:
precios = df['precio_usd'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución completa (log scale)
axes[0].hist(precios, bins=100, color='#2471A3', edgecolor='white', alpha=0.8)
axes[0].set_yscale('log')
axes[0].set_xlabel("Precio (USD)")
axes[0].set_ylabel("Frecuencia (log)")
axes[0].set_title("Distribución de precios (escala log)")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}K'))

# Zoom hasta 500K (donde está el grueso)
precios_zoom = precios[precios <= 500_000]
axes[1].hist(precios_zoom, bins=50, color='#1B4F72', edgecolor='white', alpha=0.8)
axes[1].axvline(precios.median(), color='red', linestyle='--', lw=2, label=f'Mediana: USD {precios.median():,.0f}')
axes[1].axvline(precios.mean(), color='orange', linestyle='--', lw=2, label=f'Media: USD {precios.mean():,.0f}')
axes[1].set_xlabel("Precio (USD)")
axes[1].set_ylabel("Frecuencia")
axes[1].set_title("Distribución de precios (hasta USD 500K)")
axes[1].legend(fontsize=9)
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))

plt.tight_layout()
plt.show()

q1, q3 = precios.quantile(0.25), precios.quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers_precio = precios[(precios < lower) | (precios > upper)]
print(f"\nIQR method: {len(outliers_precio):,} outliers ({len(outliers_precio)/len(precios)*100:.1f}%)")
print(f"Q1: USD {q1:,.0f} | Q3: USD {q3:,.0f} | Upper fence: USD {upper:,.0f}")


In [ ]:
[markdown]
# ---
# ## 4. Estructura del DataFrame y Tipos de Datos


In [ ]:
print("=" * 65)
print("RESUMEN DE TIPOS DE DATOS CAPTURADOS")
print("=" * 65)

categorias = {
    'Textuales': ['Precio', 'Expensas', 'Calle', 'Barrio', 'Descripción',
                  'Detalles', 'Caracteristicas', 'Tipo_Propiedad', 'Link'],
    'Numéricos continuos': ['Sup_Cubierta_m2', 'Sup_Total_m2', 'Antiguedad',
                            'Latitud', 'Longitud'],
    'Ordinales': ['Piso', 'Ambientes', 'Dormitorios', 'Banos', 'Altura'],
    'Dicotómicos (0/1)': ['Amenities', 'Losa_Central', 'Aire_Acond', 'Apto_Credito',
                          'Cochera', 'Seguridad', 'Luminoso', 'Balcon_Aterrazado', 'A_Estrenar'],
}

for tipo, cols in categorias.items():
    existing = [c for c in cols if c in df.columns]
    print(f"\n{tipo} ({len(existing)}):")
    print(f"  {', '.join(existing)}")

print(f"\nTotal: {sum(len([c for c in v if c in df.columns]) for v in categorias.values())} variables clasificadas de {df.shape[1]}")


In [ ]:
[markdown]
# ---
# ## 5. Análisis por Barrio


In [ ]:
# Top barrios
top_barrios = df['Barrio'].value_counts().head(21)

fig, ax = plt.subplots(figsize=(12, 7))
colors_bar = ['#1B4F72' if v >= 900 else '#2471A3' if v >= 700 else '#5DADE2' for v in top_barrios.values]
ax.barh(range(len(top_barrios)), top_barrios.values, color=colors_bar, edgecolor='white')
ax.set_yticks(range(len(top_barrios)))
ax.set_yticklabels(top_barrios.index, fontsize=10)
ax.set_xlabel("Cantidad de departamentos")
ax.set_title(f"Registros scrapeados por barrio (Top 21 de {df['Barrio'].nunique()} detectados)")
ax.invert_yaxis()
for i, val in enumerate(top_barrios.values):
    ax.text(val + 10, i, f'{val:,}', va='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
[markdown]
# ### 5.1 Precio mediano por m² por barrio (KPI #1)


In [ ]:
# Filtrar coordenadas válidas y precios válidos para este análisis
mask_valid = df['precio_m2'].notna() & df['precio_m2'].between(100, 15000)
df_valid = df[mask_valid].copy()

precio_m2_barrio = (df_valid.groupby('Barrio')['precio_m2']
                    .agg(['median', 'mean', 'std', 'count'])
                    .rename(columns={'median': 'Mediana_m2', 'mean': 'Media_m2',
                                     'std': 'Desvio_m2', 'count': 'N'})
                    .sort_values('Mediana_m2', ascending=False))

# Coeficiente de variación (KPI #4)
precio_m2_barrio['CV_pct'] = (precio_m2_barrio['Desvio_m2'] / precio_m2_barrio['Media_m2'] * 100).round(1)

# Filtrar barrios con al menos 50 registros
precio_m2_barrio_top = precio_m2_barrio[precio_m2_barrio['N'] >= 50].head(21)

fig, ax = plt.subplots(figsize=(13, 7))
colors_precio = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(precio_m2_barrio_top)))
bars = ax.barh(range(len(precio_m2_barrio_top)), precio_m2_barrio_top['Mediana_m2'],
               color=colors_precio, edgecolor='white')
ax.set_yticks(range(len(precio_m2_barrio_top)))
ax.set_yticklabels(precio_m2_barrio_top.index, fontsize=10)
ax.set_xlabel("Precio mediano por m² (USD)")
ax.set_title("Precio mediano del m² por barrio (barrios con n≥50)")
ax.invert_yaxis()

for i, (med, cv, n) in enumerate(zip(precio_m2_barrio_top['Mediana_m2'],
                                      precio_m2_barrio_top['CV_pct'],
                                      precio_m2_barrio_top['N'])):
    ax.text(med + 30, i, f'USD {med:,.0f}  (CV={cv:.0f}%, n={n})', va='center', fontsize=8)

plt.tight_layout()
plt.show()

print("Los barrios con mayor CV% presentan más dispersión → más oportunidades de arbitraje.")


In [ ]:
[markdown]
# ---
# ## 6. Smart Features — Variables Dicotómicas


In [ ]:
dicotomicos = ['Amenities', 'Losa_Central', 'Aire_Acond', 'Apto_Credito',
               'Cochera', 'Seguridad', 'Luminoso', 'Balcon_Aterrazado', 'A_Estrenar']

feat_data = pd.Series({col: df[col].mean() * 100 for col in dicotomicos}).sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
colors_feat = plt.cm.viridis(np.linspace(0.2, 0.8, len(feat_data)))
ax.barh(range(len(feat_data)), feat_data.values, color=colors_feat, edgecolor='white')
ax.set_yticks(range(len(feat_data)))
ax.set_yticklabels(feat_data.index, fontsize=10)
ax.set_xlabel("% del stock")
ax.set_title("Prevalencia de Smart Features en el dataset (NLP sobre descripciones)")

for i, val in enumerate(feat_data.values):
    ax.text(val + 0.5, i, f'{val:.1f}% ({int(val/100*len(df)):,})', va='center', fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
[markdown]
# ---
# ## 7. Distribución de Tipologías


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Ambientes
amb = df['Ambientes'].dropna()
amb_counts = amb.value_counts().sort_index()
amb_counts = amb_counts[amb_counts.index <= 7]
axes[0].bar(amb_counts.index.astype(int), amb_counts.values, color='#2471A3', edgecolor='white')
axes[0].set_xlabel("Ambientes")
axes[0].set_ylabel("Cantidad")
axes[0].set_title("Distribución por ambientes")
for i, (x, v) in enumerate(zip(amb_counts.index, amb_counts.values)):
    axes[0].text(x, v + 50, f'{v:,}', ha='center', fontsize=8)

# Superficie cubierta
sup = df['Sup_Cubierta_m2'].dropna()
sup_filt = sup[(sup >= 10) & (sup <= 300)]
axes[1].hist(sup_filt, bins=40, color='#1B4F72', edgecolor='white', alpha=0.8)
axes[1].axvline(sup_filt.median(), color='red', linestyle='--', lw=2, label=f'Mediana: {sup_filt.median():.0f} m²')
axes[1].set_xlabel("Superficie cubierta (m²)")
axes[1].set_ylabel("Frecuencia")
axes[1].set_title("Distribución de superficie cubierta")
axes[1].legend(fontsize=9)

# Antigüedad (filtrando errores)
ant_clean = df['Antiguedad'].dropna()
ant_clean = ant_clean[(ant_clean >= 0) & (ant_clean <= 150)]
axes[2].hist(ant_clean, bins=40, color='#2ECC71', edgecolor='white', alpha=0.8)
axes[2].axvline(ant_clean.median(), color='red', linestyle='--', lw=2, label=f'Mediana: {ant_clean.median():.0f} años')
axes[2].set_xlabel("Antigüedad (años)")
axes[2].set_ylabel("Frecuencia")
axes[2].set_title("Distribución de antigüedad")
axes[2].legend(fontsize=9)

plt.suptitle("Distribución de las principales variables del dataset", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
[markdown]
# ---
# ## 8. Mapa de Cobertura Geográfica


In [ ]:
# Filtrar solo coordenadas válidas dentro de CABA
df_geo = df[df['Latitud'].between(LAT_MIN, LAT_MAX) & df['Longitud'].between(LON_MIN, LON_MAX)].copy()

fig, ax = plt.subplots(figsize=(10, 12))
scatter = ax.scatter(df_geo['Longitud'], df_geo['Latitud'],
                     c=df_geo['precio_usd'], cmap='RdYlGn_r',
                     s=3, alpha=0.4,
                     vmin=df_geo['precio_usd'].quantile(0.05),
                     vmax=df_geo['precio_usd'].quantile(0.95))
cbar = plt.colorbar(scatter, ax=ax, shrink=0.6, label='Precio USD')
cbar.ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")
ax.set_title(f"Cobertura geográfica del scraping ({len(df_geo):,} deptos geocodificados)")
ax.set_facecolor('#f0f0f0')
plt.tight_layout()
plt.show()

print(f"Registros con coordenadas válidas en CABA: {len(df_geo):,} de {len(df):,}")


In [ ]:
[markdown]
# ---
# ## 9. Cálculo Preliminar de KPIs


In [ ]:
print("=" * 65)
print("KPIs PRELIMINARES (Pre-Entrega 1)")
print("=" * 65)

# KPI 1: Precio mediano m² por barrio (ya calculado arriba)
print("\n--- KPI 1: Precio mediano m² (top 5 barrios) ---")
for barrio, row in precio_m2_barrio_top.head(5).iterrows():
    print(f"  {barrio:<20} USD {row['Mediana_m2']:>8,.0f} /m²  (n={int(row['N'])})")

# KPI 4: Coeficiente de variación
print("\n--- KPI 4: Barrios con mayor dispersión de precios (CV%) ---")
top_cv = precio_m2_barrio[precio_m2_barrio['N'] >= 50].nlargest(5, 'CV_pct')
for barrio, row in top_cv.iterrows():
    print(f"  {barrio:<20} CV = {row['CV_pct']:.1f}%  (más oportunidades de arbitraje)")

# KPI 6: Ratio Apto Crédito por barrio
print("\n--- KPI 6: Ratio Apto Crédito (top 5 barrios) ---")
apto_credito_barrio = (df.groupby('Barrio')['Apto_Credito']
                       .agg(['mean', 'count'])
                       .rename(columns={'mean': 'ratio', 'count': 'n'})
                       .query('n >= 50')
                       .nlargest(5, 'ratio'))
for barrio, row in apto_credito_barrio.iterrows():
    print(f"  {barrio:<20} {row['ratio']*100:.1f}% apto crédito  (n={int(row['n'])})")

# Gap Estrenar vs Resto (proxy del Gap de Flipping)
print("\n--- KPI 2: Gap de Flipping (aproximación) ---")
df_gap = df_valid.copy()
df_gap['segmento'] = np.where(df_gap['A_Estrenar'] == 1, 'A estrenar', 'Usado/Otro')
gap = df_gap.groupby('segmento')['precio_m2'].median()
if len(gap) == 2:
    gap_pct = (gap['A estrenar'] - gap['Usado/Otro']) / gap['Usado/Otro'] * 100
    print(f"  Mediana m² A estrenar:  USD {gap['A estrenar']:,.0f}")
    print(f"  Mediana m² Usado/Otro:  USD {gap['Usado/Otro']:,.0f}")
    print(f"  Gap global:             {gap_pct:+.1f}%")


In [ ]:
[markdown]
# ---
# ## 10. Resumen Ejecutivo del Dataset


In [ ]:
print("=" * 65)
print("RESUMEN EJECUTIVO — DATASET ARGENPROP CABA")
print("=" * 65)

stats = {
    "Total de registros": f"{len(df):,}",
    "Variables": f"28 originales + 3 computadas (precio_usd, expensas_ars, precio_m2)",
    "Barrios cubiertos": f"{df['Barrio'].nunique()} (21 principales)",
    "Registros con precio USD": f"{df['precio_usd'].notna().sum():,} ({df['precio_usd'].notna().mean()*100:.1f}%)",
    "Registros geocodificados (CABA)": f"{coords_ok.sum():,} ({coords_ok.mean()*100:.1f}%)",
    "Precio mediano": f"USD {df['precio_usd'].median():,.0f}",
    "Precio medio": f"USD {df['precio_usd'].mean():,.0f}",
    "Superficie mediana": f"{df['Sup_Cubierta_m2'].median():.0f} m²",
    "Tipología predominante": "2 ambientes (31%)",
    "Smart Features": "9 variables dicotómicas vía NLP",
    "Fuente": "Argenprop (web scraping, abril 2026)",
    "Representatividad": "~23% del stock CABA",
}

for k, v in stats.items():
    print(f"  {k:<40} {v}")


In [ ]:
[markdown]
# ---
# ## 11. Problemas de Calidad Detectados (input para Pre-Entrega 2)
#
# | Problema | Magnitud | Acción propuesta |
# |----------|----------|------------------|
# | Nulos en Antigüedad | 38.3% | Imputar con mediana por barrio o usar variable `A_Estrenar` |
# | Nulos en Expensas | 38.5% | Evaluar si correlaciona con tipo de edificio; imputar o excluir |
# | Nulos en Dormitorios | 19.3% | Inferir desde `Ambientes` (Dormitorios ≈ Ambientes - 1) |
# | Coordenadas fuera de CABA | ~2% | Re-geocodificar con Nominatim o eliminar |
# | Antigüedad = 2026 | Puntual | Convertir año a antigüedad (2026 - valor) |
# | Precios extremos (>USD 5M) | <0.5% | Evaluar caso a caso; segmento super-lujo genuino vs error |
# | Superficies = 1 m² | Puntual | Error de carga; eliminar o imputar |


In [ ]:
print("✓ Notebook completado.")
print("✓ Dataset validado y listo para la fase de limpieza (Pre-Entrega 2).")
print(f"✓ {len(df):,} registros disponibles para análisis.")
